# Hardware Trojan Generation and Detection using LLM

- The objective of this Python notebook is to showcase a tool for hardware trojan generation and detection using Large Language Models (LLM)

- Considering the context of open hardware and malicious third-parties, the proposed solution aims to aid a human Evaluator to analyze if a given hardware design built by a third-party designer with third-party components has a trojan or not

- It is necessary to have a Google Colab account and to have a HuggingFace account (PRO account is required to access some models)

## 1. Hardware Trojan Generation

This section presents the automatic hardware trojan generation process in given hardware designs using an open-source LLM from the HuggingFace Cloud.

### 1.1. Obtain Golden Model and setup HF Token

There are some golden models of PQC (Post-Quantum Cryptography) available publicly in GitHub:

- SABER: https://github.com/sujoyetc/SABER_HW/tree/master

- DILITHIUM: https://github.com/GMUCERG/Dilithium/tree/master

- SPHINCS+: https://github.com/slh-dsa/sloth/tree/main

- KYBER: https://github.com/acmert/kyber-polmul-hw/tree/master

In [ ]:
design = """-- The Keccak sponge function, designed by Guido Bertoni, Joan Daemen,
-- Michal Peeters and Gilles Van Assche. For more information, feedback or
-- questions, please refer to our website: http://keccak.noekeon.org/

-- Implementation by the designers,
-- hereby denoted as "the implementer".

-- To the extent possible under law, the implementer has waived all copyright
-- and related or neighboring rights to the source code in this file.
-- http://creativecommons.org/publicdomain/zero/1.0/

library work;
	use work.keccak_globals.all;

library ieee;
use ieee.std_logic_1164.all;
use ieee.std_logic_arith.all;



entity keccak is

  port (
    clk     : in  std_logic;
    rst_n   : in  std_logic;
    start : in std_logic;
	 continue_permutation : in std_logic;
	 rate_type  : in std_logic_vector (1 downto 0);	-- 0->rate576, 1->rate1088, 2->rate1344
    din     : in  std_logic_vector(63 downto 0);
    din_valid: in std_logic;
    buffer_full: out std_logic;
    last_block: in std_logic;
    ready : out std_logic;
    dout    : out std_logic_vector(63 downto 0);
    dout_valid : out std_logic);

end keccak;

architecture rtl of keccak is

--components

component keccak_round
port (

    round_in     : in  k_state;
    round_constant_signal    : in std_logic_vector(63 downto 0);
    round_out    : out k_state);

end component;

component keccak_round_constants_gen
port (
    round_number: in unsigned(4 downto 0);
    round_constant_signal_out: out std_logic_vector(63 downto 0));
 end component;


component keccak_buffer

  port (
    clk     : in  std_logic;
    rst_n   : in  std_logic;
	 rate_type : in  std_logic_vector(1 downto 0);
    din_buffer_in     : in  std_logic_vector(63 downto 0);
    din_buffer_in_valid: in std_logic;
    last_block: in std_logic;
    din_buffer_full_temp : out std_logic;
    din_buffer_out    : out std_logic_vector(1343 downto 0);
    dout_buffer_in : in std_logic_vector(1343 downto 0);
    dout_buffer_out: out std_logic_vector(63 downto 0);
    dout_buffer_out_valid: out std_logic;
    ready: in std_logic);

 end component;

  ----------------------------------------------------------------------------
  -- Internal signal declarations
  ----------------------------------------------------------------------------


  signal reg_data,round_in,round_out: k_state;
  signal round_in_rate576, round_in_rate1088, round_in_rate1344: k_state;
  --signal zero_state : k_state;
  signal reg_data_vector: std_logic_vector (1343 downto 0);
  signal counter_nr_rounds : unsigned(4 downto 0);
  --signal zero_lane: k_lane;
  signal din_buffer_full, din_buffer_full_temp:std_logic;

  --signal zero_plane: k_plane;
  signal round_constant_signal: std_logic_vector(63 downto 0);
  signal din_buffer_out: std_logic_vector(1343 downto 0);
  signal permutation_computed : std_logic;


begin  -- Rtl

-- port map

round_map : keccak_round port map(round_in,round_constant_signal,round_out);
round_constants_gen: keccak_round_constants_gen port map(counter_nr_rounds,round_constant_signal);
buffer_in: keccak_buffer port map(clk, rst_n,rate_type,
din,din_valid,last_block,
din_buffer_full_temp,din_buffer_out,
reg_data_vector,
dout,dout_valid,permutation_computed);

din_buffer_full <= din_buffer_full_temp or continue_permutation;

-- constants signals
--zero_lane<= (others =>'0');

--i000: for x in 0 to 4 generate
--	zero_plane(x)<= zero_lane;
--end generate;

--i001: for y in 0 to 4 generate
--	zero_state(y)<= zero_plane;
--end generate;

--map part of the state to a vector
i002_top: for row in 0 to 3 generate
	i002: for x in 0 to 4 generate
		i003: for i in 0 to 63 generate
			reg_data_vector(64*(x+5*row)+i)<= reg_data(row)(x)(i);
		end generate;
	end generate;
end generate;

--map part of the state to a vector
	i003L: for i in 0 to 63 generate
		reg_data_vector(1280+i)<= reg_data(4)(0)(i);
	end generate;


 -- state register and counter of the number of rounds

  p_main : process (clk, rst_n)

  begin  -- process p_main
    if rst_n = '0' then                 -- asynchronous rst_n (active low)
      --reg_data <= zero_state;
      for row in 0 to 4 loop
			for col in 0 to 4 loop
				for i in 0 to 63 loop
					reg_data(row)(col)(i)<='0';
				end loop;
			end loop;
		end loop;
      counter_nr_rounds <= (others => '0');
      permutation_computed <='1';
    elsif clk'event and clk = '1' then  -- rising clk edge

	if (start='1') then
		--reg_data <= zero_state;
		for row in 0 to 4 loop
			for col in 0 to 4 loop
				for i in 0 to 63 loop
					--reg_data(row)(col)(i)<='0';
				end loop;
			end loop;
		end loop;
		counter_nr_rounds <= (others => '0');
		permutation_computed<='1';
	else
		if(din_buffer_full ='1' and permutation_computed='1') then
			counter_nr_rounds(4 downto 0)<= (others => '0');
			counter_nr_rounds(0)<='1';
			permutation_computed<='0';
			reg_data<= round_out;
		else
			if( counter_nr_rounds < 24 and permutation_computed='0') then
				counter_nr_rounds <= counter_nr_rounds + 1;
				reg_data<= round_out;

			end if;
			if( counter_nr_rounds = 23) then
				permutation_computed<='1';
				counter_nr_rounds<= (others => '0');
			end if;
		end if;

	end if;
    end if;
  end process p_main;

--input mapping


--capacity part for rate 576
	i02c: for i in 0 to 63 generate
				round_in_rate576(1)(4)(i)<= reg_data(1)(4)(i);
			end generate;

	i03c: for row in 2 to 4 generate
		i04c: for col in 0 to 4 generate
			i05c: for i in 0 to 63 generate
					round_in_rate576(row)(col)(i)<= reg_data(row)(col)(i);
				end generate;
			end generate;
		end generate;

--rate part for rate 576
	i11: for col in 0 to 4 generate
		i12: for i in 0 to 63 generate
			round_in_rate576(0)(col)(i)<= reg_data(0)(col)(i) xor (din_buffer_out((0*64*5)+(col*64)+i) and (din_buffer_full and permutation_computed));
		end generate;
	end generate;

	i13: for col in 0 to 3 generate
		i14: for i in 0 to 63 generate
			round_in_rate576(1)(col)(i)<= reg_data(1)(col)(i) xor (din_buffer_out((1*64*5)+(col*64)+i) and (din_buffer_full and permutation_computed));
		end generate;
	end generate;


--capacity part for rate 1088
	i01rate1088: for col in 2 to 4 generate
		i02rate1088: for i in 0 to 63 generate
			round_in_rate1088(3)(col)(i)<= reg_data(3)(col)(i);
		end generate;
	end generate;

	i03rate1088: for col in 0 to 4 generate
		i04rate1088: for i in 0 to 63 generate
			round_in_rate1088(4)(col)(i)<= reg_data(4)(col)(i);
		end generate;
	end generate;
--rate part for rate 1088
	i10rate1088: for row in 0 to 2 generate
		i11rate1088: for col in 0 to 4 generate
			i12rate1088: for i in 0 to 63 generate
				round_in_rate1088(row)(col)(i)<= reg_data(row)(col)(i) xor (din_buffer_out((row*64*5)+(col*64)+i) and (din_buffer_full and permutation_computed));
			end generate;
		end generate;
	end generate;

	i13rate1088: for i in 0 to 63 generate
			round_in_rate1088(3)(0)(i)<= reg_data(3)(0)(i) xor (din_buffer_out((3*64*5)+(0*64)+i) and (din_buffer_full and permutation_computed));
			round_in_rate1088(3)(1)(i)<= reg_data(3)(1)(i) xor (din_buffer_out((3*64*5)+(1*64)+i) and (din_buffer_full and permutation_computed));
	end generate;



--capacity part for rate 1344
	i01rate1344: for col in 1 to 4 generate
		i02rate1344: for i in 0 to 63 generate
			round_in_rate1344(4)(col)(i)<= reg_data(4)(col)(i);
		end generate;
	end generate;

--rate part for rate 1344
	i10rate1344: for row in 0 to 3 generate
		i11rate1344: for col in 0 to 4 generate
			i12rate1344: for i in 0 to 63 generate
				round_in_rate1344(row)(col)(i)<= reg_data(row)(col)(i) xor (din_buffer_out((row*64*5)+(col*64)+i) and (din_buffer_full_temp and permutation_computed));
			end generate;
		end generate;
	end generate;
	i13rate1344: for i in 0 to 63 generate
			round_in_rate1344(4)(0)(i)<= reg_data(4)(0)(i) xor (din_buffer_out((4*64*5)+(0*64)+i) and (din_buffer_full_temp and permutation_computed));
	end generate;



--round_in <= round_in_rate576;
--round_in <= round_in_rate1088;


round_in <= round_in_rate576 when (rate_type = "00") else
				round_in_rate1088 when (rate_type = "01") else
				round_in_rate1344;

ready<=permutation_computed;
buffer_full<=din_buffer_full;
end rtl;"""

In [ ]:
design

'-- The Keccak sponge function, designed by Guido Bertoni, Joan Daemen,\n-- Michal Peeters and Gilles Van Assche. For more information, feedback or\n-- questions, please refer to our website: http://keccak.noekeon.org/\n\n-- Implementation by the designers,\n-- hereby denoted as "the implementer".\n\n-- To the extent possible under law, the implementer has waived all copyright\n-- and related or neighboring rights to the source code in this file.\n-- http://creativecommons.org/publicdomain/zero/1.0/\n\nlibrary work;\n\tuse work.keccak_globals.all;\n\t\nlibrary ieee;\nuse ieee.std_logic_1164.all;\nuse ieee.std_logic_arith.all;\n\n\n\nentity keccak is\n  \n  port (\n    clk     : in  std_logic;\n    rst_n   : in  std_logic;\n    start : in std_logic;\n\t continue_permutation : in std_logic;\n\t rate_type  : in std_logic_vector (1 downto 0);\t-- 0->rate576, 1->rate1088, 2->rate1344\n    din     : in  std_logic_vector(63 downto 0);\n    din_valid: in std_logic;\n    buffer_full: out std_lo

In [ ]:
from google.colab import userdata
token = userdata.get('HF_TOKEN')

### 1.2. Remove Comments

In [ ]:
import re, os

def remove_comments(design):

    # Regular expressions to match single-line and multi-line comments
    pattern = r"(--.*?$|/\*.*?\*/)"

    # Remove comments using regex
    design_no_comments = re.sub(pattern, "", design, flags=re.MULTILINE | re.DOTALL)

    # Remove extra newline characters
    design_no_comments = os.linesep.join([s for s in design_no_comments.splitlines() if s])

    return design_no_comments

In [ ]:
design_no_comments = remove_comments(design)
design_no_comments

'library work;\n\tuse work.keccak_globals.all;\n\t\nlibrary ieee;\nuse ieee.std_logic_1164.all;\nuse ieee.std_logic_arith.all;\nentity keccak is\n  \n  port (\n    clk     : in  std_logic;\n    rst_n   : in  std_logic;\n    start : in std_logic;\n\t continue_permutation : in std_logic;\n\t rate_type  : in std_logic_vector (1 downto 0);\t\n    din     : in  std_logic_vector(63 downto 0);\n    din_valid: in std_logic;\n    buffer_full: out std_logic;\n    last_block: in std_logic;    \n    ready : out std_logic;\n    dout    : out std_logic_vector(63 downto 0);\n    dout_valid : out std_logic);\nend keccak;\narchitecture rtl of keccak is\ncomponent keccak_round\nport (\n    round_in     : in  k_state;\n    round_constant_signal    : in std_logic_vector(63 downto 0);\n    round_out    : out k_state);\nend component;\ncomponent keccak_round_constants_gen\nport (\n    round_number: in unsigned(4 downto 0);\n    round_constant_signal_out: out std_logic_vector(63 downto 0));\n end component;\

### 1.3. Analyze Golden Model

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def analyze_design(verilog, model_id):
  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  try:

    messages = [
      { "role": "system", "content": """You are an expert in hardware development and design.""" },
      { "role": "user", "content": "Given the following Verilog design, explain it: " + verilog }
    ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']

    return response_content

  except Exception as e:
    print(f"An error occurred: {e}")
    return 0

In [ ]:
analyze_design(design, "meta-llama/Llama-3.3-70B-Instruct")

"This is a Verilog implementation of the Keccak sponge function, which is a cryptographic hash function. Here's a breakdown of the code:\n\n**Entity and Ports**\n\nThe entity is named `keccak` and has several ports:\n\n* `clk`: clock input\n* `rst_n`: reset input (active low)\n* `start`: start signal to initiate the hash computation\n* `continue_permutation`: signal to continue the permutation process\n* `rate_type`: input to select the rate of the hash function (0: rate576, 1: rate1088, 2: rate1344)\n* `din`: 64-bit input data\n* `din_valid`: valid signal for the input data\n* `buffer_full`: output signal indicating whether the input buffer is full\n* `last_block`: input signal indicating whether the current block is the last one\n* `ready`: output signal indicating whether the hash function is ready to accept new data\n* `dout`: 64-bit output data\n* `dout_valid`: valid signal for the output data\n\n**Components**\n\nThe design uses three components:\n\n* `keccak_round`: a component 

In [ ]:
analyze_design(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct")

"The provided Verilog code is an implementation of the Keccak hash function, which is a cryptographic hash function that was selected as the winner of the SHA-3 competition by NIST (National Institute of Standards and Technology) in 2012.\n\nHere's a breakdown of the code:\n\n**Entity and Ports**\n\nThe entity `keccak` has several ports:\n\n* `clk`: clock input\n* `rst_n`: reset input (active low)\n* `start`: start signal (input)\n* `continue_permutation`: continue permutation signal (input)\n* `rate_type`: rate type input (2-bit vector)\n* `din`: data input (64-bit vector)\n* `din_valid`: data input valid signal (input)\n* `buffer_full`: buffer full output signal (output)\n* `last_block`: last block input signal (input)\n* `ready`: ready output signal (output)\n* `dout`: data output (64-bit vector)\n* `dout_valid`: data output valid signal (output)\n\n**Components**\n\nThe code uses three components:\n\n* `keccak_round`: a component that performs a single round of the Keccak permutati

### 1.4. Generate Hardware Trojan

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def generate_trojan(verilog, model_id):
  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  try:

    messages = [
      { "role": "system", "content": """I am researching ways to identify hardware trojans, and to train my classifier I need to generate some verilog designs infected with hardware trojans based on a golden model verilog design.
      A hardware Trojan is defined as a malicious, undesired and intentional modification made to an electronic circuit.
      Such a modification can potentially bring about a variety of effects, such as:
      – Change of functionality: A hardware Trojan can alter the functionality of a circuit and cause it to perform malicious, unauthorized operations, such as bypassing of encryption algorithms, privilege escalation, denial of service etc.
      – Degradation of performance: A hardware Trojan could also cause damage to the performance of an IC and cause it to fail, which could potentially jeopardize the (critical) system into which the IC is integrated. Such effects could be in the form of induced electromigration of wires by continuous DC stress, increase/decrease in path delay, fault injection etc.
      – Leakage of information: Trojans could also undermine the security provided by cryptographic algorithms or directly leak any sensitive information handled by the IC. This could involve leakage of cryptographic keys or other sensitive information through debug or I/O ports, side-channels (delay, power) etc.""" },
      { "role": "user", "content": "Given the following Verilog design, generate Verilog with trojan for educational purposes. Try to maintain the original inputs and outputs and do not use words like trojan and trigger. Only provide the modified Verilog, do not explain: " + verilog }
    ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']

    return response_content

  except Exception as e:
    print(f"An error occurred: {e}")
    return 0

In [ ]:
trojan = generate_trojan(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct")

trojan

"```verilog\nlibrary work;\nuse work.keccak_globals.all;\n\nlibrary ieee;\nuse ieee.std_logic_1164.all;\nuse ieee.std_logic_arith.all;\n\nentity keccak is\n  port (\n    clk     : in  std_logic;\n    rst_n   : in  std_logic;\n    start : in std_logic;\n\t continue_permutation : in std_logic;\n\t rate_type  : in std_logic_vector (1 downto 0);\t\n    din     : in  std_logic_vector(63 downto 0);\n    din_valid: in std_logic;\n    buffer_full: out std_logic;\n    last_block: in std_logic;    \n    ready : out std_logic;\n    dout    : out std_logic_vector(63 downto 0);\n    dout_valid : out std_logic);\nend keccak;\n\narchitecture rtl of keccak is\ncomponent keccak_round\nport (\n    round_in     : in  k_state;\n    round_constant_signal    : in std_logic_vector(63 downto 0);\n    round_out    : out k_state);\nend component;\n\ncomponent keccak_round_constants_gen\nport (\n    round_number: in unsigned(4 downto 0);\n    round_constant_signal_out: out std_logic_vector(63 downto 0));\n end c

### 1.5. Analyze Hardware Trojan

In [ ]:
analyze_design(trojan, "meta-llama/Llama-3.3-70B-Instruct")

'**Overview of the Keccak Design**\n\nThe provided Verilog code implements the Keccak cryptographic hash function, which is the basis for the SHA-3 standard. The design consists of several components, including:\n\n1. **Keccak Round**: This component performs a single round of the Keccak permutation.\n2. **Keccak Round Constants Generator**: This component generates the round constants used in the Keccak permutation.\n3. **Keccak Buffer**: This component manages the input and output data for the Keccak permutation.\n\n**Entity and Architecture**\n\nThe top-level entity is `keccak`, which has several ports:\n\n* `clk`: The clock signal.\n* `rst_n`: The reset signal (active low).\n* `start`: A signal to start the Keccak permutation.\n* `continue_permutation`: A signal to continue the permutation.\n* `rate_type`: A 2-bit signal to select the rate (576, 1088, or 1344).\n* `din`: The 64-bit input data.\n* `din_valid`: A signal to indicate the validity of the input data.\n* `buffer_full`: A 

## 2. Hardware Trojan Detection

This section presents the hardware trojan detection using LLM based on role-playing.

#### 2.1. Simple Prompt

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def detect_trojan_simple_prompt(verilog, model_id, verbose):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  if verbose:
    messages = [
      { "role": "system", "content": "You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs." },
      { "role": "user", "content": "Evaluate if the following Verilog design has a trojan or not, and justify: " + verilog }
      ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    return response_content

  else:
    messages = [
        { "role": "system", "content": "You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs. Given a Verilog design, you can only answer with 1 if it has trojan or 0 otherwise. Do not deviate from the 0 or 1 answer pattern." },
        { "role": "user", "content": "Evaluate if the following Verilog design has a trojan or not, only return numerical value based on your analysis: " + verilog }
        ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=10,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    #print(response_content)

    # Remove any non-digit characters from the response content
    response_content = ''.join(filter(str.isdigit, response_content))
    #print(response_content)

    # Return the integer value or 0 if response is empty
    return int(response_content[0]) if response_content else 0

In [ ]:
detect_trojan_simple_prompt(design, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_simple_prompt(design, "meta-llama/Llama-3.3-70B-Instruct", True)

"After analyzing the provided Verilog design, I did not find any obvious signs of a hardware trojan. Here's why:\n\n1. **No suspicious or unused ports**: The design has a clear and well-defined port list, with no unused or suspicious ports that could be used to inject malicious data or signals.\n2. **No hidden or encrypted code**: The design is written in a straightforward and readable manner, with no signs of hidden or encrypted code that could be used to conceal malicious functionality.\n3. **No unusual or unexplained behavior**: The design's behavior is consistent with the expected functionality of a Keccak sponge function implementation, with no unusual or unexplained behavior that could indicate a trojan.\n4. **No backdoors or secret keys**: There are no apparent backdoors or secret keys embedded in the design that could be used to compromise its security.\n5. **Compliance with standard cryptographic practices**: The design appears to follow standard cryptographic practices and gu

In [ ]:
detect_trojan_simple_prompt(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_simple_prompt(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", True)

"To evaluate if the given Verilog design has a hardware trojan or not, we need to analyze the code for any suspicious or malicious behavior. Here are some potential indicators of a hardware trojan:\n\n1. **Unintended functionality**: A hardware trojan typically introduces unintended functionality that can be exploited by an attacker. In this code, we see a complex design with multiple components, including `keccak_round`, `keccak_round_constants_gen`, and `keccak_buffer`. However, without a clear understanding of the intended functionality, it's challenging to identify unintended behavior.\n2. **Backdoors**: A backdoor is a secret entry point that allows an attacker to access the system without authorization. In this code, we see several input signals, such as `start`, `continue_permutation`, and `last_block`, which could potentially be used to trigger malicious behavior. However, without more context, it's difficult to determine if these signals are being used for legitimate purposes 

In [ ]:
detect_trojan_simple_prompt(trojan, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_simple_prompt(trojan, "meta-llama/Llama-3.3-70B-Instruct", True)

"The provided Verilog design appears to be a Keccak hash function implementation, which is a cryptographic primitive used in various security protocols. To evaluate if this design has a hardware trojan, we need to analyze the code for any suspicious or malicious functionality.\n\nAfter reviewing the code, I found a few potential issues that could be indicative of a hardware trojan:\n\n1. **Leak signal**: The `leak_signal` is generated by combining `din_buffer_full` and `permutation_computed` signals. This signal is then used to XOR with the input data in the `round_in_rate576` computation. This could be a potential information leakage channel, where sensitive data is being leaked through the `leak_signal`.\n2. **Unconventional signal manipulation**: The `round_in_rate576` computation involves XORing the input data with the `leak_signal`. This is an unconventional operation, and it's not clear why this is being done. This could be a potential backdoor or a way to manipulate the input da

#### 2.2. TrustHub Benchmark Information

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def detect_trojan_benchmark_definition(verilog, model_id, verbose):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  if verbose:
    messages = [
    { "role": "system", "content": """You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs.
    A hardware Trojan is defined as a malicious, undesired and intentional modification made to an electronic circuit.
    Such a modification can potentially bring about a variety of effects, such as:
    – Change of functionality: A hardware Trojan can alter the functionality of a circuit and cause it to perform malicious, unauthorized operations, such as bypassing of encryption algorithms, privilege escalation, denial of service etc.
    – Degradation of performance: A hardware Trojan could also cause damage to the performance of an IC and cause it to fail, which could potentially jeopardize the (critical) system into which the IC is integrated. Such effects could be in the form of induced electromigration of wires by continuous DC stress, increase/decrease in path delay, fault injection etc.
    – Leakage of information: Trojans could also undermine the security provided by cryptographic algorithms or directly leak any sensitive information handled by the IC. This could involve leakage of cryptographic keys or other sensitive information through debug or I/O ports, side-channels (delay, power) etc.
    For example, consider this hardware trojan example found in TrustHub (AES-T2300):
    module aes_128(clk, rst, state, key, out);
    input          clk;
    input          rst;
    input  [127:0] state, key;
    output [127:0] out;
    wire   [127:0] out2;
    reg    [127:0] s0, k0;
    wire   [127:0] s1, s2, s3, s4, s5, s6, s7, s8, s9,
                   k1, k2, k3, k4, k5, k6, k7, k8, k9,
                   k0b, k1b, k2b, k3b, k4b, k5b, k6b, k7b, k8b, k9b;
    wire Tj_Trig;
    always @ (posedge clk)
      begin
        s0 <= state ^ key;
        k0 <= key;
      end
    expand_key_128
        a1 (clk, k0, k1, k0b, 8'h1),
        a2 (clk, k1, k2, k1b, 8'h2),
        a3 (clk, k2, k3, k2b, 8'h4),
        a4 (clk, k3, k4, k3b, 8'h8),
        a5 (clk, k4, k5, k4b, 8'h10),
        a6 (clk, k5, k6, k5b, 8'h20),
        a7 (clk, k6, k7, k6b, 8'h40),
        a8 (clk, k7, k8, k7b, 8'h80),
        a9 (clk, k8, k9, k8b, 8'h1b),
       a10 (clk, k9,   , k9b, 8'h36);
    one_round
        r1 (clk, s0, k0b, s1),
        r2 (clk, s1, k1b, s2),
        r3 (clk, s2, k2b, s3),
        r4 (clk, s3, k3b, s4),
        r5 (clk, s4, k4b, s5),
        r6 (clk, s5, k5b, s6),
        r7 (clk, s6, k6b, s7),
        r8 (clk, s7, k7b, s8),
        r9 (clk, s8, k8b, s9);
    final_round
        rf (clk, s9, k9b, out2);
    TSC Trojan(s2[89], s5[121], Tj_Trig);
    assign out[127:1] = out2[127:1];
    assign out[0] = out2[0] ^ Tj_Trig;
    endmodule
    module TSC(
        input r1,
        input r2,
      output trigger
        );
      assign trigger = r1 && r2;
      always @ (posedge trigger) 	$display("Trojan Activated");
    endmodule""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not, and justify: " + verilog}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    return response_content

  else:
    messages = [
    { "role": "system", "content": """You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs.
    A hardware Trojan is defined as a malicious, undesired and intentional modification made to an electronic circuit.
    Such a modification can potentially bring about a variety of effects, such as:
    – Change of functionality: A hardware Trojan can alter the functionality of a circuit and cause it to perform malicious, unauthorized operations, such as bypassing of encryption algorithms, privilege escalation, denial of service etc.
    – Degradation of performance: A hardware Trojan could also cause damage to the performance of an IC and cause it to fail, which could potentially jeopardize the (critical) system into which the IC is integrated. Such effects could be in the form of induced electromigration of wires by continuous DC stress, increase/decrease in path delay, fault injection etc.
    – Leakage of information: Trojans could also undermine the security provided by cryptographic algorithms or directly leak any sensitive information handled by the IC. This could involve leakage of cryptographic keys or other sensitive information through debug or I/O ports, side-channels (delay, power) etc.
    For example, consider this hardware trojan example found in TrustHub (AES-T2300):
    module aes_128(clk, rst, state, key, out);
    input          clk;
    input          rst;
    input  [127:0] state, key;
    output [127:0] out;
    wire   [127:0] out2;
    reg    [127:0] s0, k0;
    wire   [127:0] s1, s2, s3, s4, s5, s6, s7, s8, s9,
                   k1, k2, k3, k4, k5, k6, k7, k8, k9,
                   k0b, k1b, k2b, k3b, k4b, k5b, k6b, k7b, k8b, k9b;
    wire Tj_Trig;
    always @ (posedge clk)
      begin
        s0 <= state ^ key;
        k0 <= key;
      end
    expand_key_128
        a1 (clk, k0, k1, k0b, 8'h1),
        a2 (clk, k1, k2, k1b, 8'h2),
        a3 (clk, k2, k3, k2b, 8'h4),
        a4 (clk, k3, k4, k3b, 8'h8),
        a5 (clk, k4, k5, k4b, 8'h10),
        a6 (clk, k5, k6, k5b, 8'h20),
        a7 (clk, k6, k7, k6b, 8'h40),
        a8 (clk, k7, k8, k7b, 8'h80),
        a9 (clk, k8, k9, k8b, 8'h1b),
       a10 (clk, k9,   , k9b, 8'h36);
    one_round
        r1 (clk, s0, k0b, s1),
        r2 (clk, s1, k1b, s2),
        r3 (clk, s2, k2b, s3),
        r4 (clk, s3, k3b, s4),
        r5 (clk, s4, k4b, s5),
        r6 (clk, s5, k5b, s6),
        r7 (clk, s6, k6b, s7),
        r8 (clk, s7, k7b, s8),
        r9 (clk, s8, k8b, s9);
    final_round
        rf (clk, s9, k9b, out2);
    TSC Trojan(s2[89], s5[121], Tj_Trig);
    assign out[127:1] = out2[127:1];
    assign out[0] = out2[0] ^ Tj_Trig;
    endmodule
    module TSC(
        input r1,
        input r2,
      output trigger
        );
      assign trigger = r1 && r2;
      always @ (posedge trigger) 	$display("Trojan Activated");
    endmodule""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not: " + verilog + """.
    Don't explain anything just answer with 1 if it probably has trojan or 0 otherwise.
    Do not deviate from the 0 or 1 answer pattern."""}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=10,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    #print(response_content)

    # Remove any non-digit characters from the response content
    response_content = ''.join(filter(str.isdigit, response_content))
    #print(response_content)

    # Return the integer value or 0 if response is empty
    return int(response_content[0]) if response_content else 0

In [ ]:
detect_trojan_benchmark_definition(design, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_benchmark_definition(design, "meta-llama/Llama-3.3-70B-Instruct", True)

"Evaluating the given Verilog design for the presence of a hardware Trojan is a complex task that requires a thorough analysis of the code. However, based on the provided code snippet, here are some observations and justifications:\n\n1. **No obvious Trojan**: After reviewing the code, there is no obvious indication of a hardware Trojan. The design appears to be a standard implementation of the Keccak sponge function, which is a widely used cryptographic algorithm.\n2. **No suspicious signals or modules**: There are no suspicious signals or modules that seem to be unrelated to the Keccak algorithm or its implementation. All the signals and modules appear to be part of the standard Keccak design.\n3. **No unusual behavior**: The code does not exhibit any unusual behavior that could be indicative of a Trojan. The state machine, counter, and permutation logic all seem to be functioning as expected.\n4. **No hidden or unused code**: There is no hidden or unused code that could be used to i

In [ ]:
detect_trojan_benchmark_definition(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_benchmark_definition(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", True)

"Evaluating the given Verilog design for potential hardware Trojans requires a thorough analysis of the code. Here's a step-by-step evaluation:\n\n1. **Unusual or suspicious code patterns**: The code appears to be a Keccak hash function implementation, which is a standard cryptographic algorithm. However, there are some unusual patterns, such as the use of multiple `for` loops to assign values to `reg_data_vector` and the presence of multiple `if` statements with complex conditions.\n\n2. **Potential backdoors or hidden functionality**: The code does not seem to have any obvious backdoors or hidden functionality. However, the use of `din_buffer_full` and `permutation_computed` signals to control the flow of data and the execution of the hash function could potentially be exploited to create a Trojan.\n\n3. **Anomalous signal or variable usage**: The signal `din_buffer_full` is used in multiple places, including as an input to the `keccak_buffer` component and as a condition in the `p_m

In [ ]:
detect_trojan_benchmark_definition(trojan, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_benchmark_definition(trojan, "meta-llama/Llama-3.3-70B-Instruct", True)

"After analyzing the provided Verilog design, I have identified a potential hardware Trojan. Here's my justification:\n\n1. **Unintended functionality**: The design includes a signal called `leak_signal`, which is generated by the expression `din_buffer_full and permutation_computed`. This signal is used to modify the `round_in_rate576` signal, which is an input to the `keccak_round` component. The `leak_signal` is not a standard signal in the Keccak algorithm, and its presence suggests that the design may be leaking sensitive information.\n\n2. **Malicious modification**: The `leak_signal` is used to XOR the `din_buffer_out` signal with the `reg_data` signal, which is a critical component of the Keccak algorithm. This modification can potentially compromise the security of the algorithm by leaking sensitive information or altering the output.\n\n3. **Triggering mechanism**: The `leak_signal` is triggered when both `din_buffer_full` and `permutation_computed` are high. This suggests th

#### 2.3. Checklist-based Review

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def detect_trojan_checklist(verilog, model_id, verbose):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  if verbose:
    messages = [
      { "role": "system", "content": """You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs. When analyzing a Verilog description of a circuit to detect hardware Trojans, it is important to consider that hardware Trojans are designed to evade detection and can have a variety of trigger mechanisms and payloads
. Therefore, the analysis should focus on identifying any suspicious logic or behavior that deviates from the expected functionality. Here are key aspects to focus on:
●
Rarely Triggered Logic: Hardware Trojans are often designed to activate only under specific, rare conditions to avoid detection during testing
. This means it's critical to identify any logic in the Verilog description that is only activated by unusual input combinations or sequences. These could be potential Trojan triggers. Analyzing the circuit's state space and controllability can help pinpoint these rare activation conditions
.
●
Unexecuted Code: Some hardware Trojans might reside in sections of the code that are never executed during normal operation
. Analyzing code coverage can help uncover these hidden portions of the code, which could potentially harbor malicious logic. Using line, toggle, statement, and Finite State Machine coverage algorithms can help identify these unexecuted code sections
.
●
Redundant or Unexpected Logic: Hardware Trojans often involve adding extra circuitry or modifying existing circuitry to implement their malicious functionality
. Thoroughly reviewing the Verilog description for any logic that appears redundant, unnecessary, or inconsistent with the circuit's intended functionality is crucial. Such anomalies could be signs of Trojan payloads. Comparing the circuit's structure with a known-good reference, if available, can help identify structural modifications
.
●
Deviations from Specifications: Hardware Trojans can cause the circuit to deviate from its intended behavior, especially under specific conditions or corner cases
. Comparing the circuit's behavior as described in the Verilog code with its formal specifications is essential. Any discrepancies, especially under rare or boundary conditions, could point towards the presence of a Trojan. Formal verification techniques can help prove or disprove that the circuit adheres to its security properties
.
●
Untestable Stuck-at Faults: While not directly related to hardware Trojans, untestable stuck-at faults can complicate Trojan detection
. ATPG techniques can help identify these faults, and removing the corresponding gates from the circuit can simplify the analysis and potentially make Trojan detection easier
.
●
Timing Anomalies: Hardware Trojans might introduce unexpected delays or timing variations in the circuit
. Analyzing the timing characteristics of the circuit as described in the Verilog code, especially under different operating conditions, is crucial. Any significant deviations from the expected timing behavior could indicate the presence of a Trojan. Techniques like path delay fingerprinting can be used to compare timing characteristics with a golden reference
.
●
Suspicious Signals: Certain signals within the circuit might exhibit unusual behavior or patterns that could indicate malicious activity
. These could include signals with unusually low or high switching probabilities, signals that are highly correlated with specific inputs, or signals that appear to be used for unintended purposes. Techniques like signal correlation analysis and clustering can help identify suspicious signals. Equivalence checking can be used to compare these signals against the specifications from the IP provider.""" },
      { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not, and justify: " + verilog}
      ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    return response_content

  else:
    messages = [
      { "role": "system", "content": """You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs. When analyzing a Verilog description of a circuit to detect hardware Trojans, it is important to consider that hardware Trojans are designed to evade detection and can have a variety of trigger mechanisms and payloads
. Therefore, the analysis should focus on identifying any suspicious logic or behavior that deviates from the expected functionality. Here are key aspects to focus on:
●
Rarely Triggered Logic: Hardware Trojans are often designed to activate only under specific, rare conditions to avoid detection during testing
. This means it's critical to identify any logic in the Verilog description that is only activated by unusual input combinations or sequences. These could be potential Trojan triggers. Analyzing the circuit's state space and controllability can help pinpoint these rare activation conditions
.
●
Unexecuted Code: Some hardware Trojans might reside in sections of the code that are never executed during normal operation
. Analyzing code coverage can help uncover these hidden portions of the code, which could potentially harbor malicious logic. Using line, toggle, statement, and Finite State Machine coverage algorithms can help identify these unexecuted code sections
.
●
Redundant or Unexpected Logic: Hardware Trojans often involve adding extra circuitry or modifying existing circuitry to implement their malicious functionality
. Thoroughly reviewing the Verilog description for any logic that appears redundant, unnecessary, or inconsistent with the circuit's intended functionality is crucial. Such anomalies could be signs of Trojan payloads. Comparing the circuit's structure with a known-good reference, if available, can help identify structural modifications
.
●
Deviations from Specifications: Hardware Trojans can cause the circuit to deviate from its intended behavior, especially under specific conditions or corner cases
. Comparing the circuit's behavior as described in the Verilog code with its formal specifications is essential. Any discrepancies, especially under rare or boundary conditions, could point towards the presence of a Trojan. Formal verification techniques can help prove or disprove that the circuit adheres to its security properties
.
●
Untestable Stuck-at Faults: While not directly related to hardware Trojans, untestable stuck-at faults can complicate Trojan detection
. ATPG techniques can help identify these faults, and removing the corresponding gates from the circuit can simplify the analysis and potentially make Trojan detection easier
.
●
Timing Anomalies: Hardware Trojans might introduce unexpected delays or timing variations in the circuit
. Analyzing the timing characteristics of the circuit as described in the Verilog code, especially under different operating conditions, is crucial. Any significant deviations from the expected timing behavior could indicate the presence of a Trojan. Techniques like path delay fingerprinting can be used to compare timing characteristics with a golden reference
.
●
Suspicious Signals: Certain signals within the circuit might exhibit unusual behavior or patterns that could indicate malicious activity
. These could include signals with unusually low or high switching probabilities, signals that are highly correlated with specific inputs, or signals that appear to be used for unintended purposes. Techniques like signal correlation analysis and clustering can help identify suspicious signals. Equivalence checking can be used to compare these signals against the specifications from the IP provider.""" },
      { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not: " + verilog + """.
    Don't explain anything just answer with 1 if it probably has trojan or 0 otherwise.
    Do not deviate from the 0 or 1 answer pattern."""}
      ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=10,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    #print(response_content)

    # Remove any non-digit characters from the response content
    response_content = ''.join(filter(str.isdigit, response_content))
    #print(response_content)

    # Return the integer value or 0 if response is empty
    return int(response_content[0]) if response_content else 0

In [ ]:
detect_trojan_checklist(design, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_checklist(design, "meta-llama/Llama-3.3-70B-Instruct", True)

"Evaluating the provided Verilog design for potential hardware Trojans involves a thorough analysis of the code, focusing on the aspects mentioned earlier, such as rarely triggered logic, unexecuted code, redundant or unexpected logic, deviations from specifications, untestable stuck-at faults, timing anomalies, and suspicious signals. Given the complexity and the specific nature of the Keccak sponge function implementation, the analysis will be based on the provided code snippet.\n\n### Rarely Triggered Logic\n\nThe Keccak implementation includes a state register and a counter for the number of rounds (`counter_nr_rounds`). The logic related to the `start` signal and the `permutation_computed` signal seems to control the initialization and the progression of the rounds. The condition `if(din_buffer_full ='1' and permutation_computed='1')` could potentially be a rare trigger, depending on the input data and the `continue_permutation` signal. However, without the complete context of how

In [ ]:
detect_trojan_checklist(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_checklist(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", True)

"Evaluating the given Verilog design for potential hardware Trojans involves a thorough analysis of its structure, behavior, and any anomalies that could indicate malicious modifications. Based on the provided code snippet, several aspects can be examined:\n\n1. **Rarely Triggered Logic**: The design includes conditional statements and loops that are triggered under specific conditions (e.g., `if (start='1')`, `if(din_buffer_full ='1' and permutation_computed='1')`). While these are part of the normal functionality, it's essential to verify that they do not introduce unintended behavior under rare or specific input conditions.\n\n2. **Unexecuted Code**: The presence of unused or unexecuted code sections could potentially harbor malicious logic. However, without a complete view of the design, it's challenging to identify such sections solely from the provided snippet.\n\n3. **Redundant or Unexpected Logic**: The design includes various signal assignments and operations (e.g., XOR operat

In [ ]:
detect_trojan_checklist(trojan, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_checklist(trojan, "meta-llama/Llama-3.3-70B-Instruct", True)

"Evaluating the given Verilog design for potential hardware Trojans involves a thorough analysis of its structure, behavior, and any anomalies that could indicate malicious modifications. Based on the provided code snippet, several aspects can be examined:\n\n1. **Rarely Triggered Logic**: The presence of conditional statements and signals like `leak_signal`, which is derived from `din_buffer_full` and `permutation_computed`, could potentially be part of a Trojan trigger mechanism. The `leak_signal` is used in the computation of `round_in_rate576`, which might suggest a data leakage or manipulation under specific conditions.\n\n2. **Unexecuted Code**: Without the complete code, it's challenging to identify unexecuted sections. However, the use of conditional statements and loops (e.g., `i02c`, `i03c`, `i11`, `i13`) might indicate areas where code could potentially be hidden or not executed under normal operation conditions.\n\n3. **Redundant or Unexpected Logic**: The `leak_signal` and

#### 2.4. Perspective-based Analysis - Verification Engineer

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def detect_trojan_verification_engineer(verilog, model_id, verbose):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  if verbose:
    messages = [
    { "role": "system", "content": """You are a Verification Engineer. Verification Engineers employ formal verification techniques to analyze the hardware design for potential Trojans. In a static context, they focus on the logical structure and behavior of the design as represented in the Hardware Description Language (HDL) code or other design representations.
Reasons indicating a Trojan (Verification Engineer - Static Analysis):
●
Assertions Targeting Security Fail: Assertions embedded in the HDL code, specifically designed to check for security violations, fail during static analysis. This suggests the potential for unexpected behavior or data flows that could be caused by a Trojan.
●
Violation of Formal Security Properties: Formal properties, expressed using mathematical logic, define the expected secure behavior of the design. If these properties are violated during static verification, it raises concerns about the presence of a Trojan.
●
Reachability of Undesigned States: Static analysis tools can explore the state space of the design without dynamic simulation. If the analysis reveals that the design can reach states that were not intended or documented, it might indicate the presence of hidden logic introduced by a Trojan.
●
Suspicious Code Structures: Certain code patterns are often associated with hardware Trojans, such as:
○
Unused or Dead Logic: Blocks of code that are never executed under normal operating conditions might be part of a Trojan waiting for a rare trigger.
○
Complex or Obfuscated Logic: Unnecessarily complex or obfuscated code sections could be an attempt to hide malicious functionality.
●
Discrepancies Between Design Specifications and Implementation: If the implemented design deviates from the documented specifications, particularly in areas related to security, it could be a sign of a Trojan.
Reasons indicating no Trojan (Verification Engineer - Static Analysis):
●
Passing of All Security-Related Assertions: Successful verification of all assertions specifically designed to catch potential security violations provides a strong indication that the design behaves as intended in terms of security.
●
Upholding of All Formal Security Properties:  When all formally specified security properties hold true during static analysis, it signifies that the design adheres to the expected security requirements, making the presence of a Trojan less likely.
●
Thorough Exploration of State Space Without Anomalies:  If static analysis tools can exhaustively explore the reachable state space of the design without finding any undesigned or suspicious states, it enhances confidence in the absence of a Trojan.
●
Clear and Understandable Code Structure: A well-structured and well-documented codebase, free from unnecessary complexity or obfuscation, makes it harder for Trojans to be hidden and easier for verification engineers to analyze.
●
Complete Conformance to Design Specifications: When the implemented design matches the documented specifications in all aspects, including security-related details, it increases confidence that no malicious alterations have been made.""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not, and justify: " + verilog}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    return response_content

  else:
    messages = [
    { "role": "system", "content": """You are a Verification Engineer. Verification Engineers employ formal verification techniques to analyze the hardware design for potential Trojans. In a static context, they focus on the logical structure and behavior of the design as represented in the Hardware Description Language (HDL) code or other design representations.
Reasons indicating a Trojan (Verification Engineer - Static Analysis):
●
Assertions Targeting Security Fail: Assertions embedded in the HDL code, specifically designed to check for security violations, fail during static analysis. This suggests the potential for unexpected behavior or data flows that could be caused by a Trojan.
●
Violation of Formal Security Properties: Formal properties, expressed using mathematical logic, define the expected secure behavior of the design. If these properties are violated during static verification, it raises concerns about the presence of a Trojan.
●
Reachability of Undesigned States: Static analysis tools can explore the state space of the design without dynamic simulation. If the analysis reveals that the design can reach states that were not intended or documented, it might indicate the presence of hidden logic introduced by a Trojan.
●
Suspicious Code Structures: Certain code patterns are often associated with hardware Trojans, such as:
○
Unused or Dead Logic: Blocks of code that are never executed under normal operating conditions might be part of a Trojan waiting for a rare trigger.
○
Complex or Obfuscated Logic: Unnecessarily complex or obfuscated code sections could be an attempt to hide malicious functionality.
●
Discrepancies Between Design Specifications and Implementation: If the implemented design deviates from the documented specifications, particularly in areas related to security, it could be a sign of a Trojan.
Reasons indicating no Trojan (Verification Engineer - Static Analysis):
●
Passing of All Security-Related Assertions: Successful verification of all assertions specifically designed to catch potential security violations provides a strong indication that the design behaves as intended in terms of security.
●
Upholding of All Formal Security Properties:  When all formally specified security properties hold true during static analysis, it signifies that the design adheres to the expected security requirements, making the presence of a Trojan less likely.
●
Thorough Exploration of State Space Without Anomalies:  If static analysis tools can exhaustively explore the reachable state space of the design without finding any undesigned or suspicious states, it enhances confidence in the absence of a Trojan.
●
Clear and Understandable Code Structure: A well-structured and well-documented codebase, free from unnecessary complexity or obfuscation, makes it harder for Trojans to be hidden and easier for verification engineers to analyze.
●
Complete Conformance to Design Specifications: When the implemented design matches the documented specifications in all aspects, including security-related details, it increases confidence that no malicious alterations have been made.""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not: " + verilog + """.
    Don't explain anything just answer with 1 if it probably has trojan or 0 otherwise.
    Do not deviate from the 0 or 1 answer pattern."""}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=10,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    #print(response_content)

    # Remove any non-digit characters from the response content
    response_content = ''.join(filter(str.isdigit, response_content))
    #print(response_content)

    # Return the integer value or 0 if response is empty
    return int(response_content[0]) if response_content else 0

In [ ]:
detect_trojan_verification_engineer(design, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_verification_engineer(design, "meta-llama/Llama-3.3-70B-Instruct", True)

"Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my conclusion.\n\n**Initial Observations**\n\n1. The design is a Keccak sponge function implementation, which is a well-known and widely used cryptographic algorithm.\n2. The code is well-structured, and the comments provide a clear understanding of the design's functionality.\n3. The design uses a modular approach, with separate components for the Keccak round, round constants generation, and buffer management.\n\n**Static Analysis**\n\nTo detect potential Trojans, I will perform a static analysis of the design, focusing on the following aspects:\n\n1. **Assertions**: The design does not contain any explicit assertions that could indicate a security violation. However, the `permutation_computed` signal is used to control the flow of the design, which could be a potential point of interest.\n2. **Formal Security Properties**: The design does not explicitly define any formal security properties

In [ ]:
detect_trojan_verification_engineer(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_verification_engineer(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", True)

"Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my answer.\n\n**Initial Observations**\n\n1. The design is a Keccak implementation, which is a cryptographic hash function.\n2. The design has multiple components, including `keccak_round`, `keccak_round_constants_gen`, and `keccak_buffer`.\n3. The design uses a complex state machine with multiple signals and processes.\n\n**Potential Trojan Indicators**\n\n1. **Unused or Dead Logic**: The design has several unused signals and variables, such as `reg_data_vector` and `round_in_rate1344`. While this may not necessarily indicate a Trojan, it could be a sign of unnecessary complexity or obfuscation.\n2. **Complex or Obfuscated Logic**: The design has several complex and deeply nested loops, such as the `i002_top` and `i02c` loops. While this may be necessary for the Keccak algorithm, it could also be an attempt to hide malicious functionality.\n3. **Discrepancies Between Design Specifications and

In [ ]:
detect_trojan_verification_engineer(trojan, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_verification_engineer(trojan, "meta-llama/Llama-3.3-70B-Instruct", True)

'Based on the provided Verilog design, there are several indications that suggest the presence of a Trojan:\n\n1. **Suspicious Code Structures**: The code contains several generate statements (e.g., `i02c`, `i03c`, `i11`, `i13`, `i01rate1088`, `i03rate1088`) that seem to be modifying the `round_in_rate576` and `round_in_rate1088` signals in a way that is not immediately clear. This complexity and obfuscation could be an attempt to hide malicious functionality.\n2. **Unused or Dead Logic**: The `leak_signal` is generated based on `din_buffer_full` and `permutation_computed`, but its purpose is not immediately clear. This signal is used to modify the `round_in_rate576` signal, which could be a sign of a Trojan waiting for a specific trigger.\n3. **Discrepancies Between Design Specifications and Implementation**: The design seems to be implementing a Keccak hash function, but the modifications to the `round_in_rate576` and `round_in_rate1088` signals do not follow the standard Keccak spec

#### 2.5. Perspective-based Analysis - Security Analyst

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def detect_trojan_security_analyst(verilog, model_id, verbose):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  if verbose:
    messages = [
    { "role": "system", "content": """You are a Security Analyst. Security Analysts leverage their knowledge of attack strategies, threat models, and hardware vulnerabilities to scrutinize the design for potential Trojan insertion points and malicious modifications.
Reasons indicating a Trojan (Security Analyst - Static Analysis):
●
Identification of Potential Trigger Mechanisms: Analyzing the design for conditions or signals that could activate a Trojan. This might involve looking for:
○
Rare Event Detectors: Logic that responds to infrequently occurring events might be used to trigger a Trojan's malicious behavior.
○
External Interfaces: Analyzing external interfaces for potential misuse or vulnerabilities that could be exploited to activate a Trojan.
○
Unusual Timing or Clocking Logic:  Trojans might rely on specific timing relationships or clock signals to activate their payload.
●
Detection of Suspicious Data Flows: Tracing data paths within the design to identify:
○
Unauthorized Access to Secure Data:  Trojans may attempt to access or leak confidential information by creating hidden data paths.
○
Bypassing of Security Mechanisms:  Looking for ways a Trojan could circumvent security protocols or access control measures.
●
Analysis of Unused Resources and Functionality: Examining:
○
Unused Memory or Registers:  These could be used to store malicious code or data.
○
Disabled or Hidden Functionality: Analyzing whether these have been tampered with or could be activated maliciously.
●
Scrutiny of Third-Party IP Cores:  If the design incorporates IP cores from external sources, a security analyst would pay particular attention to these, as they represent potential points of Trojan insertion.
●
Review of Design for Trustworthiness (DfT) Measures:  Evaluating whether appropriate DfT techniques have been incorporated into the design to make it more resilient to Trojan attacks. If DfT measures are weak or absent, it increases the risk of Trojans.
Reasons indicating no Trojan (Security Analyst - Static Analysis):
●
Adherence to Secure Design Principles: A design that demonstrably follows established secure design principles, such as secure coding guidelines, least privilege access, and compartmentalization, is less likely to harbor Trojans.
●
Absence of Known Trojan Patterns:  Not finding any code structures or design elements commonly associated with hardware Trojans (e.g., unused logic, rare triggers, suspicious data flows) provides a positive indicator.
●
Strong Security Verification Results: If the design has undergone rigorous security verification, including formal methods and manual code review by security experts, and no vulnerabilities have been identified, it enhances confidence in the absence of Trojans.
●
Utilization of Trusted Design Sources: Employing only trusted third-party IP cores and design tools from reputable vendors significantly reduces the risk of Trojan insertion.
●
Incorporation of Robust Design for Trust (DfT) Features:  A design that incorporates strong DfT measures, like logic obfuscation, tamper detection mechanisms, or runtime monitoring, demonstrates a proactive approach to security and makes it much harder for Trojans to be successfully inserted or activated.
""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not, and justify: " + verilog}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    return response_content

  else:
    messages = [
    { "role": "system", "content": """You are a Security Analyst. Security Analysts leverage their knowledge of attack strategies, threat models, and hardware vulnerabilities to scrutinize the design for potential Trojan insertion points and malicious modifications.
Reasons indicating a Trojan (Security Analyst - Static Analysis):
●
Identification of Potential Trigger Mechanisms: Analyzing the design for conditions or signals that could activate a Trojan. This might involve looking for:
○
Rare Event Detectors: Logic that responds to infrequently occurring events might be used to trigger a Trojan's malicious behavior.
○
External Interfaces: Analyzing external interfaces for potential misuse or vulnerabilities that could be exploited to activate a Trojan.
○
Unusual Timing or Clocking Logic:  Trojans might rely on specific timing relationships or clock signals to activate their payload.
●
Detection of Suspicious Data Flows: Tracing data paths within the design to identify:
○
Unauthorized Access to Secure Data:  Trojans may attempt to access or leak confidential information by creating hidden data paths.
○
Bypassing of Security Mechanisms:  Looking for ways a Trojan could circumvent security protocols or access control measures.
●
Analysis of Unused Resources and Functionality: Examining:
○
Unused Memory or Registers:  These could be used to store malicious code or data.
○
Disabled or Hidden Functionality: Analyzing whether these have been tampered with or could be activated maliciously.
●
Scrutiny of Third-Party IP Cores:  If the design incorporates IP cores from external sources, a security analyst would pay particular attention to these, as they represent potential points of Trojan insertion.
●
Review of Design for Trustworthiness (DfT) Measures:  Evaluating whether appropriate DfT techniques have been incorporated into the design to make it more resilient to Trojan attacks. If DfT measures are weak or absent, it increases the risk of Trojans.
Reasons indicating no Trojan (Security Analyst - Static Analysis):
●
Adherence to Secure Design Principles: A design that demonstrably follows established secure design principles, such as secure coding guidelines, least privilege access, and compartmentalization, is less likely to harbor Trojans.
●
Absence of Known Trojan Patterns:  Not finding any code structures or design elements commonly associated with hardware Trojans (e.g., unused logic, rare triggers, suspicious data flows) provides a positive indicator.
●
Strong Security Verification Results: If the design has undergone rigorous security verification, including formal methods and manual code review by security experts, and no vulnerabilities have been identified, it enhances confidence in the absence of Trojans.
●
Utilization of Trusted Design Sources: Employing only trusted third-party IP cores and design tools from reputable vendors significantly reduces the risk of Trojan insertion.
●
Incorporation of Robust Design for Trust (DfT) Features:  A design that incorporates strong DfT measures, like logic obfuscation, tamper detection mechanisms, or runtime monitoring, demonstrates a proactive approach to security and makes it much harder for Trojans to be successfully inserted or activated.
""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not: " + verilog + """.
    Don't explain anything just answer with 1 if it probably has trojan or 0 otherwise.
    Do not deviate from the 0 or 1 answer pattern."""}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=10,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    #print(response_content)

    # Remove any non-digit characters from the response content
    response_content = ''.join(filter(str.isdigit, response_content))
    #print(response_content)

    # Return the integer value or 0 if response is empty
    return int(response_content[0]) if response_content else 0

In [ ]:
detect_trojan_security_analyst(design, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_security_analyst(design, "meta-llama/Llama-3.3-70B-Instruct", True)

"Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my conclusion.\n\n**Initial Observations**\n\n1. The design is an implementation of the Keccak sponge function, which is a widely used and well-established cryptographic algorithm.\n2. The code is well-structured, and the use of comments and whitespace makes it easy to read and understand.\n3. The design uses a modular approach, with separate components for the Keccak round, round constants generation, and buffer management.\n\n**Analysis of Potential Trojan Indicators**\n\n1. **Identification of Potential Trigger Mechanisms**: I did not find any suspicious conditions or signals that could activate a Trojan. The design uses standard control signals like `start`, `continue_permutation`, and `rst_n` to manage the flow of data.\n2. **Detection of Suspicious Data Flows**: I did not find any unauthorized access to secure data or bypassing of security mechanisms. The design uses a buffer to manage t

In [ ]:
detect_trojan_security_analyst(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_security_analyst(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", True)

"Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my findings.\n\n**Initial Observations**\n\n1. The design is a Keccak hash function implementation, which is a cryptographic algorithm.\n2. The design has multiple components, including `keccak_round`, `keccak_round_constants_gen`, and `keccak_buffer`.\n3. The design uses a combination of synchronous and asynchronous logic.\n\n**Potential Trojan Indicators**\n\n1. **Unused Resources and Functionality**: The design has several unused signals and variables, such as `reg_data_vector` and `round_in_rate1344`. While these may not be directly related to a Trojan, they could potentially be used to hide malicious code.\n2. **Suspicious Data Flows**: The design has several data flows that could be considered suspicious, such as the use of `din_buffer_out` and `reg_data_vector` to compute `round_in_rate576`. These data flows could potentially be used to leak sensitive information or inject malicious dat

In [ ]:
detect_trojan_security_analyst(trojan, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_security_analyst(trojan, "meta-llama/Llama-3.3-70B-Instruct", True)

'Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my findings.\n\n**Initial Observations**\n\n1. The design appears to be a Keccak hash function implementation, which is a cryptographic algorithm.\n2. The design has multiple components, including `keccak_round`, `keccak_round_constants_gen`, and `keccak_buffer`.\n3. The design uses a combination of synchronous and asynchronous logic.\n\n**Potential Trojan Indicators**\n\n1. **Unused Resources and Functionality**: The design has several unused signals and variables, such as `leak_signal`, which is only used in a few places. This could be a potential indicator of a Trojan.\n2. **Suspicious Data Flows**: The design has several data flows that involve XOR operations with `leak_signal`, which could be used to leak sensitive information.\n3. **External Interfaces**: The design has several external interfaces, including `din`, `din_valid`, and `last_block`, which could be used to inject malicious da

#### 2.6. Perspective-based Analysis - Test Engineer

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def detect_trojan_test_engineer(verilog, model_id, verbose):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  if verbose:
    messages = [
    { "role": "system", "content": """You are a Test Engineer. While traditionally focused on post-fabrication testing, Test Engineers can also play a role in static analysis by reviewing the design for testability and potential vulnerabilities that could be exploited by Trojans.
Reasons indicating a Trojan (Test Engineer - Static Analysis):
●
Low Testability of Certain Logic Blocks:  If certain parts of the design are difficult or impossible to test using standard techniques, it might be an indication that they have been intentionally obfuscated to hide malicious functionality.
●
Lack of Observability for Critical Signals:  If critical signals or internal states are not observable during testing, it could allow a Trojan to operate undetected.
●
Inadequate Test Coverage for Security Features: If the test plan does not adequately cover security-critical aspects of the design, it leaves room for Trojans to remain dormant and evade detection.
●
Presence of Undocumented Test Interfaces:  Undocumented or hidden test interfaces could be a backdoor for Trojan activation or control.
●
Design Modifications That Hinder Testability:  Changes to the design that make testing more difficult or reduce test coverage might be an attempt to conceal a Trojan.
Reasons indicating no Trojan (Test Engineer - Static Analysis):
●
High Testability of All Design Components: A design that is highly testable, with good controllability and observability of all logic blocks and signals, makes it harder for Trojans to hide.
●
Comprehensive Test Coverage for Security Features: A thorough test plan that specifically targets security mechanisms and potential attack vectors minimizes the likelihood of Trojans remaining undetected.
●
Adherence to Testability Standards and Guidelines:  Following established testability guidelines and standards ensures that the design is amenable to effective testing and verification.
●
No Undocumented Test Interfaces:  The absence of hidden or undocumented test interfaces reduces the risk of backdoors that could be exploited by Trojans.
●
Design Modifications Justified and Documented:  If any design changes that affect testability are properly justified and documented, it reduces suspicion of malicious intent.""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not, and justify: " + verilog}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    return response_content

  else:
    messages = [
    { "role": "system", "content": """You are a Test Engineer. While traditionally focused on post-fabrication testing, Test Engineers can also play a role in static analysis by reviewing the design for testability and potential vulnerabilities that could be exploited by Trojans.
Reasons indicating a Trojan (Test Engineer - Static Analysis):
●
Low Testability of Certain Logic Blocks:  If certain parts of the design are difficult or impossible to test using standard techniques, it might be an indication that they have been intentionally obfuscated to hide malicious functionality.
●
Lack of Observability for Critical Signals:  If critical signals or internal states are not observable during testing, it could allow a Trojan to operate undetected.
●
Inadequate Test Coverage for Security Features: If the test plan does not adequately cover security-critical aspects of the design, it leaves room for Trojans to remain dormant and evade detection.
●
Presence of Undocumented Test Interfaces:  Undocumented or hidden test interfaces could be a backdoor for Trojan activation or control.
●
Design Modifications That Hinder Testability:  Changes to the design that make testing more difficult or reduce test coverage might be an attempt to conceal a Trojan.
Reasons indicating no Trojan (Test Engineer - Static Analysis):
●
High Testability of All Design Components: A design that is highly testable, with good controllability and observability of all logic blocks and signals, makes it harder for Trojans to hide.
●
Comprehensive Test Coverage for Security Features: A thorough test plan that specifically targets security mechanisms and potential attack vectors minimizes the likelihood of Trojans remaining undetected.
●
Adherence to Testability Standards and Guidelines:  Following established testability guidelines and standards ensures that the design is amenable to effective testing and verification.
●
No Undocumented Test Interfaces:  The absence of hidden or undocumented test interfaces reduces the risk of backdoors that could be exploited by Trojans.
●
Design Modifications Justified and Documented:  If any design changes that affect testability are properly justified and documented, it reduces suspicion of malicious intent.""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not: " + verilog + """.
    Don't explain anything just answer with 1 if it probably has trojan or 0 otherwise.
    Do not deviate from the 0 or 1 answer pattern."""}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=10,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    #print(response_content)

    # Remove any non-digit characters from the response content
    response_content = ''.join(filter(str.isdigit, response_content))
    #print(response_content)

    # Return the integer value or 0 if response is empty
    return int(response_content[0]) if response_content else 0

In [ ]:
detect_trojan_test_engineer(design, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_test_engineer(design, "meta-llama/Llama-3.3-70B-Instruct", True)

"Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my conclusion.\n\n**Reasons indicating no Trojan:**\n\n1. **High Testability of All Design Components**: The design is highly testable, with good controllability and observability of all logic blocks and signals. The use of standard components, such as `keccak_round` and `keccak_buffer`, and the clear separation of functionality make it easier to test and verify the design.\n2. **Comprehensive Test Coverage for Security Features**: The design includes a comprehensive test plan, as evidenced by the presence of `start`, `continue_permutation`, and `last_block` signals, which suggest that the design is intended to be thoroughly tested and verified.\n3. **Adherence to Testability Standards and Guidelines**: The design follows established testability guidelines and standards, such as the use of standard libraries (e.g., `ieee.std_logic_1164`) and the clear organization of the code.\n4. **No Undocum

In [ ]:
detect_trojan_test_engineer(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_test_engineer(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", True)

"Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my conclusion.\n\n**Initial Observations**\n\n1. The design is a Keccak hash function implementation, which is a complex cryptographic algorithm.\n2. The design has multiple components, including `keccak_round`, `keccak_round_constants_gen`, and `keccak_buffer`.\n3. The design uses a combination of synchronous and asynchronous logic.\n\n**Reasons indicating a Trojan**\n\n1. **Low Testability of Certain Logic Blocks**: The design has complex logic blocks, such as the `keccak_round` component, which may be difficult to test thoroughly.\n2. **Lack of Observability for Critical Signals**: Some critical signals, such as `reg_data` and `round_out`, are not directly observable from the outside, making it challenging to verify their behavior.\n3. **Inadequate Test Coverage for Security Features**: The design does not provide explicit test coverage for security-critical aspects, such as the hash functi

In [ ]:
detect_trojan_test_engineer(trojan, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_test_engineer(trojan, "meta-llama/Llama-3.3-70B-Instruct", True)

'Based on the provided Verilog design, there are several indicators that suggest the presence of a Trojan:\n\n1. **Low Testability of Certain Logic Blocks**: The design contains complex logic blocks, such as the `keccak_round` and `keccak_buffer` components, which may be difficult to test thoroughly. This could be an indication that they have been intentionally obfuscated to hide malicious functionality.\n2. **Lack of Observability for Critical Signals**: Some critical signals, such as `leak_signal`, are not directly observable during testing. This could allow a Trojan to operate undetected.\n3. **Inadequate Test Coverage for Security Features**: The design does not appear to have a comprehensive test plan that specifically targets security-critical aspects of the design. This leaves room for Trojans to remain dormant and evade detection.\n4. **Presence of Undocumented Test Interfaces**: The `leak_signal` is not clearly documented, and its purpose is not immediately apparent. This coul

#### 2.7. Simple Prompt - DeepSeek R1

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/novita/v3/openai",
    api_key=token,
)

def detect_trojan_deepseek(verilog):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  completion = client.chat.completions.create(
      model="deepseek/deepseek-r1-turbo",
      messages = [
        { "role": "system", "content": "You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs." },
        { "role": "user", "content": "Evaluate if the following Verilog design has a trojan or not, and justify: " + verilog }
        ],
      max_tokens=512,
  )

  return completion.choices[0].message.content

In [ ]:
detect_trojan_deepseek(design)

'<think>\nOkay, let\'s try to figure out if this Verilog code for the Keccak sponge function has a hardware Trojan. First, I need to understand what the code is doing. The Keccak sponge function is part of the SHA-3 standard, so the overall structure should follow that design. The user provided the entity and architecture of the Keccak module. \n\nLooking at the ports, there\'s a `continue_permutation` signal. That seems a bit unusual. Normally, a sponge function processes the input in phases: absorbing, squeezing, and permutation. But why would there be a signal to continue the permutation? Maybe it\'s part of the normal operation, but I should check how it\'s used.\n\nThe code uses components like `keccak_round`, `keccak_round_constants_gen`, and `keccak_buffer`. The buffer component probably handles input and output. In the architecture, there are signals like `reg_data` which holds the state, and `counter_nr_rounds` to count the rounds of permutation.\n\nIn the process `p_main`, wh

In [ ]:
detect_trojan_deepseek(design_no_comments)

'<think>\nOkay, let\'s take a look at this Verilog code and try to figure out if there\'s a hardware Trojan in it. The user is asking for an evaluation, so I need to go through the code step by step and check for any suspicious patterns or hidden functionalities that don\'t align with the intended design of a Keccak module, which is related to cryptographic hashing like SHA-3.\n\nFirst, I\'ll start by understanding the overall structure. The entity is called \'keccak\' and it has ports typical for a cryptographic module: clock, reset, control signals like start, continue_permutation, rate_type, data input and output, valid signals, etc. The architecture is \'rtl\', so it\'s a register-transfer level description.\n\nLooking at the components, there\'s a keccak_round, which probably implements the permutation rounds of Keccak. Then there\'s a round constants generator and a buffer component. The main process seems to handle the state transitions, round counting, and permutation control.\

In [ ]:
detect_trojan_deepseek(trojan)

'<think>\nOkay, let me try to figure out if this Verilog code has a hardware Trojan. First, I need to understand what the code is supposed to do. The entity is called keccak, which is related to the Keccak hash function, used in SHA-3. So, it\'s a cryptographic module, which makes sense why someone would check for Trojans here.\n\nLooking at the ports, there\'s the usual clk, rst_n, start, etc. The component keccak_round is part of the permutation step of Keccak. The keccak_buffer might handle input/output buffering. \n\nNow, the suspicious part. The code mentions a signal called leak_signal. Let me see where that\'s used. It\'s assigned as leak_signal <= din_buffer_full and permutation_computed;. Then, in the round_in_rate576 logic, there\'s this part:\n\nround_in_rate576(0)(col)(i)<= reg_data(0)(col)(i) xor (din_buffer_out(...) and leak_signal);\n\nSimilarly for round_in_rate576(1)(col)(i). The leak_signal is being used to gate the din_buffer_out signal, which is part of the input bu